In [0]:
%sql
-- ============================================================
-- GOLD LAYER - GLOBAL SUPERSTORE
-- ============================================================

-- ------------------------------------------------------------
-- 1. Create Gold schema
-- ------------------------------------------------------------

CREATE SCHEMA IF NOT EXISTS pooja.gold;


-- ------------------------------------------------------------
-- 2. Drop existing Gold table
-- ------------------------------------------------------------

DROP TABLE IF EXISTS pooja.gold.gold;


-- ------------------------------------------------------------
-- 3. Create Gold table
-- ------------------------------------------------------------
-- One table containing all business-level information
-- aggregated from the Silver layer.
-- ------------------------------------------------------------

CREATE TABLE pooja.gold.gold
USING DELTA
AS

SELECT

    -- --------------------------------------------------------
    -- Date Information
    -- --------------------------------------------------------

    Order_Year,
    Order_Month,
    Order_Quarter,
    Order_Year_Month,

    -- --------------------------------------------------------
    -- Location Information
    -- --------------------------------------------------------

    Region,
    Country,
    State,
    City,

    -- --------------------------------------------------------
    -- Customer / Product Information
    -- --------------------------------------------------------

    Segment,
    Category,
    Sub_Category,

    -- --------------------------------------------------------
    -- Business Metrics
    -- --------------------------------------------------------

    COUNT(DISTINCT Order_ID) AS Total_Orders,

    SUM(Quantity) AS Total_Quantity,

    ROUND(SUM(Sales), 2) AS Total_Sales,

    ROUND(SUM(Shipping_Cost), 2) AS Total_Shipping_Cost,

    ROUND(AVG(Discount), 2) AS Average_Discount,

    ROUND(SUM(Profit), 2) AS Total_Profit,

    -- --------------------------------------------------------
    -- Profit Margin
    -- --------------------------------------------------------

    ROUND(
        CASE
            WHEN SUM(Sales) != 0
            THEN (SUM(Profit) / SUM(Sales)) * 100
            ELSE 0
        END,
        2
    ) AS Profit_Margin

FROM delta.`/Volumes/pooja/default/pooja/silver/global_superstore`

GROUP BY

    Order_Year,
    Order_Month,
    Order_Quarter,
    Order_Year_Month,

    Region,
    Country,
    State,
    City,

    Segment,
    Category,
    Sub_Category;


-- ------------------------------------------------------------
-- 4. View Gold Table
-- ------------------------------------------------------------

SELECT *
FROM pooja.gold.gold;


-- ------------------------------------------------------------
-- 5. Check Gold Record Count
-- ------------------------------------------------------------

SELECT COUNT(*) AS Gold_Record_Count
FROM pooja.gold.gold;


-- ------------------------------------------------------------
-- 6. Business Summary
-- ------------------------------------------------------------

SELECT

    SUM(Total_Sales) AS Total_Sales,

    SUM(Total_Profit) AS Total_Profit,

    SUM(Total_Quantity) AS Total_Quantity,

    SUM(Total_Orders) AS Total_Orders,

    SUM(Total_Shipping_Cost) AS Total_Shipping_Cost,

    ROUND(
        CASE
            WHEN SUM(Total_Sales) != 0
            THEN
                (SUM(Total_Profit) / SUM(Total_Sales)) * 100
            ELSE 0
        END,
        2
    ) AS Overall_Profit_Margin

FROM pooja.gold.gold;



Total_Sales,Total_Profit,Total_Quantity,Total_Orders,Total_Shipping_Cost,Overall_Profit_Margin
1.2617755690000335E7,1466767.0799999952,198414,47608,1352688.8599999954,11.62
